<a href="https://colab.research.google.com/github/Aditya-Raj-Kaushik/Vision-Transformer/blob/main/Vision_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import random

In [3]:
torch.__version__

'2.10.0+cu128'

In [4]:
torchvision.__version__

'0.25.0+cu128'

In [5]:
!nvidia-smi

Sun May 24 10:14:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
torch.cuda.is_available()

True

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
print(f"Using Device: {device}")

Using Device: cuda


In [9]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)

In [1]:
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 3e-4
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_DIM = 256
NUM_HEADS = 8
DEPTH = 6
MLP_DIM = 512
DROP_RATE = 0.1

In [10]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

In [11]:
train_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform
)

100%|██████████| 170M/170M [00:05<00:00, 29.5MB/s]


In [12]:
test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform
)

In [13]:
train_data

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

In [14]:
len(train_data)

50000

In [15]:
len(test_data)

10000

In [16]:
train_loader = DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=BATCH_SIZE, shuffle=True)

In [17]:
print(f"Dataloader: {train_loader, test_loader}")
print(f"Length of Trainloader: {len(train_loader)}")
print(f"Length of Testloader: {len(test_loader)}")

Dataloader: (<torch.utils.data.dataloader.DataLoader object at 0x7ce64f96e4b0>, <torch.utils.data.dataloader.DataLoader object at 0x7ce64f96f290>)
Length of Trainloader: 391
Length of Testloader: 79


In [18]:
class PatchEmbedding(nn.Module):
  def __init__(self, patch_size, in_channels, embed_dim):
    super().__init__()
    self.patch_size = patch_size
    self.embed_dim = embed_dim
    self.proj = nn.Conv2d(in_channels=in_channels, out_channels=embed_dim, kernel_size=patch_size, stride=patch_size)

    num_patches = (IMAGE_SIZE // patch_size) ** 2
    self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
    self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

  def forward(self, x: torch.Tensor):
    B = x.size(0)
    x = self.proj(x)
    x = x.flatten(2).transpose(1,2)
    cls_token = self.cls_token.expand(B, -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    x = x + self.pos_embed
    return x


In [19]:
class MLP(nn.Module):
  def __init__(self, in_features, hidden_features, out_features, drop_rate=0.1):
    super().__init__()
    self.fc1 = nn.Linear(in_features=in_features, out_features=hidden_features)
    self.fc2 = nn.Linear(in_features=hidden_features, out_features=in_features)
    self.drop = nn.Dropout(drop_rate)

    def forward(self, x):
      x = self.dropout(F.gelu(self.fc1(x)))
      x = self.fc2(x)
      return x

In [20]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, drop_rate):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            dropout=drop_rate,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(embed_dim)

        self.mlp = MLP(embed_dim, mlp_dim, drop_rate)

    def forward(self, x):
        x = x + self.attn(
            self.norm1(x),
            self.norm1(x),
            self.norm1(x)
        )[0]

        x = x + self.mlp(self.norm2(x))

        return x